# E-Commerce Customer Segmentation (RFM) & Market Basket Analysis

---

**Author:** Raditya Zaki Athaya  
**Notebook:** `01_data_cleaning_eda.ipynb` — Data Cleaning & Exploratory Data Analysis

---

## Project Overview

Proyek ini bertujuan untuk mengubah data mentah transaksi e-commerce menjadi *actionable insights* yang dapat mendukung keputusan bisnis secara langsung. Analisis dibagi menjadi dua tahap utama:

1. **RFM Analysis:** Melakukan segmentasi pelanggan berdasarkan perilaku transaksi (*Recency, Frequency, Monetary*) untuk membantu tim marketing merancang kampanye retensi yang lebih personal dan efisien.
2. **Market Basket Analysis:** Mengidentifikasi asosiasi antar produk yang sering dibeli bersamaan untuk mengoptimalkan strategi *cross-selling* dan penempatan inventaris.

Notebook ini adalah **tahap pertama** yang fokus pada pembersihan data dan pemahaman awal terhadap dataset sebelum analisis lanjutan dilakukan.

## 1. Import Library & Load Data

Memuat semua library yang dibutuhkan dan melakukan *initial inspection* terhadap dataset. Langkah ini penting untuk memahami struktur data sebelum melakukan transformasi apapun.

**Dataset:** `ecommerce.csv` — berisi data transaksi e-commerce dengan 8 kolom:
`InvoiceNo`, `StockCode`, `Description`, `Quantity`, `InvoiceDate`, `UnitPrice`, `CustomerID`, `Country`.

In [1]:
# ── Standard Libraries ───────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

# ── Global Config ────────────────────────────────────────────────────────────
warnings.filterwarnings('ignore')                         # suppress non-critical warnings
pd.set_option('display.float_format', '{:,.2f}'.format)  # format angka float
sns.set_theme(style='whitegrid', palette='muted')         # consistent plot style

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [2]:
# ── Load Raw Dataset ─────────────────────────────────────────────────────────
RAW_PATH = '../data/raw/ecommerce.csv'

df = pd.read_csv(
    RAW_PATH,
    encoding='ISO-8859-1',              # encoding umum untuk dataset UK-based e-commerce
    dtype={'CustomerID': str,           # baca sebagai string agar aman saat filtering
           'InvoiceNo': str}
)

print(f"Dataset loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")
print("-" * 50)

# Tampilkan 5 baris pertama
df.head()

Dataset loaded: 541,909 rows x 8 columns
--------------------------------------------------


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850,United Kingdom


In [3]:
# ── Dataset Info ─────────────────────────────────────────────────────────────
print("=" * 55)
print(" DATASET INFO")
print("=" * 55)
df.info()

# ── Missing Value Summary ────────────────────────────────────────────────────
print("\n" + "=" * 55)
print(" MISSING VALUES PER COLUMN")
print("=" * 55)
missing     = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_summary = pd.DataFrame({
    'Missing Count': missing,
    'Missing (%)': missing_pct
})
print(missing_summary[missing_summary['Missing Count'] > 0])

# ── Descriptive Statistics ───────────────────────────────────────────────────
print("\n" + "=" * 55)
print(" DESCRIPTIVE STATISTICS")
print("=" * 55)
df.describe()

 DATASET INFO
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  object 
 7   Country      541909 non-null  object 
dtypes: float64(1), int64(1), object(6)
memory usage: 33.1+ MB

 MISSING VALUES PER COLUMN
             Missing Count  Missing (%)
Description           1454         0.27
CustomerID          135080        24.93

 DESCRIPTIVE STATISTICS


,Quantity,UnitPrice
count,"541,909.00","541,909.00"
mean,9.55,4.61
std,218.08,96.76
min,"-80,995.00","-11,062.06"
25%,1.00,1.25
50%,3.00,2.08
75%,10.00,4.13
max,"80,995.00","38,970.00"
